In [1]:
# Week 3 Day 6 — U-Net Architecture
# Goal: Build complete U-Net with Dice Loss

import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("✓ Setup complete")

Device: cuda
✓ Setup complete


In [2]:
# Cell 2 — Complete U-Net

import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels,
                               kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels,
                               kernel_size=3, padding=1)
        self.relu  = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        # ── Encoder ──
        self.enc1 = ConvBlock(in_channels, 16)
        self.enc2 = ConvBlock(16, 32)
        self.enc3 = ConvBlock(32, 64)
        self.enc4 = ConvBlock(64, 128)
        self.pool = nn.MaxPool2d(2, 2)

        # ── Bottleneck ──
        self.bottleneck = ConvBlock(128, 256)

        # ── Decoder ──
        self.upsample = nn.Upsample(scale_factor=2,
                                     mode='bilinear',
                                     align_corners=True)
        self.dec4 = ConvBlock(256 + 128, 128)
        self.dec3 = ConvBlock(128 +  64,  64)
        self.dec2 = ConvBlock( 64 +  32,  32)
        self.dec1 = ConvBlock( 32 +  16,  16)

        # ── Output ──
        self.output  = nn.Conv2d(16, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decoder
        d4 = self.dec4(torch.cat([self.upsample(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.upsample(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.upsample(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.upsample(d2), e1], dim=1))

        # Output
        return self.sigmoid(self.output(d1))

# Test U-Net
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = UNet(in_channels=1, out_channels=1).to(device)

# Simulate CT slice
ct_input = torch.zeros(1, 1, 256, 256).to(device)

print(f"Device: {device}")
print(f"Input shape: {ct_input.shape}")

with torch.no_grad():
    output = model(ct_input)

print(f"Output shape: {output.shape}")
print(f"\n── Shape journey ──")
print(f"Input:      [1,   1, 256, 256]")
print(f"Enc1:       [1,  16, 256, 256]")
print(f"Enc2:       [1,  32, 128, 128]")
print(f"Enc3:       [1,  64,  64,  64]")
print(f"Enc4:       [1, 128,  32,  32]")
print(f"Bottleneck: [1, 256,  16,  16]")
print(f"Dec4:       [1, 128,  32,  32]")
print(f"Dec3:       [1,  64,  64,  64]")
print(f"Dec2:       [1,  32, 128, 128]")
print(f"Dec1:       [1,  16, 256, 256]")
print(f"Output:     [1,   1, 256, 256]")
print(f"\n✓ U-Net built successfully")

Device: cuda
Input shape: torch.Size([1, 1, 256, 256])
Output shape: torch.Size([1, 1, 256, 256])

── Shape journey ──
Input:      [1,   1, 256, 256]
Enc1:       [1,  16, 256, 256]
Enc2:       [1,  32, 128, 128]
Enc3:       [1,  64,  64,  64]
Enc4:       [1, 128,  32,  32]
Bottleneck: [1, 256,  16,  16]
Dec4:       [1, 128,  32,  32]
Dec3:       [1,  64,  64,  64]
Dec2:       [1,  32, 128, 128]
Dec1:       [1,  16, 256, 256]
Output:     [1,   1, 256, 256]

✓ U-Net built successfully


In [4]:
# Cell 3 — Dice Loss
# Measures overlap between prediction and mask

import torch
import torch.nn as nn

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, prediction, target):
        # Flatten tensors
        pred   = prediction.view(-1)
        tgt    = target.view(-1)

        # Calculate intersection
        intersection = (pred * tgt).sum()

        # Calculate Dice score
        dice = (2. * intersection + self.smooth) / \
               (pred.sum() + tgt.sum() + self.smooth)

        # Dice Loss = 1 - Dice score
        return 1 - dice

# Test Dice Loss
dice_loss = DiceLoss()

print("── Dice Loss scenarios ──")

print("\nScenario 1 — Perfect overlap:")
prediction = torch.tensor([1.0, 1.0, 0.0, 0.0])
target     = torch.tensor([1.0, 1.0, 0.0, 0.0])
loss = dice_loss(prediction, target)
print(f"Prediction: {prediction.tolist()}")
print(f"Target:     {target.tolist()}")
print(f"Dice Loss:  {loss.item():.4f}")

print("\nScenario 2 — No overlap:")
prediction = torch.tensor([0.0, 0.0, 1.0, 1.0])
target     = torch.tensor([1.0, 1.0, 0.0, 0.0])
loss = dice_loss(prediction, target)
print(f"Prediction: {prediction.tolist()}")
print(f"Target:     {target.tolist()}")
print(f"Dice Loss:  {loss.item():.4f}")

print("\nScenario 3 — Partial overlap:")
prediction = torch.tensor([1.0, 0.0, 0.0, 0.0])
target     = torch.tensor([1.0, 1.0, 0.0, 0.0])
loss = dice_loss(prediction, target)
print(f"Prediction: {prediction.tolist()}")
print(f"Target:     {target.tolist()}")
print(f"Dice Loss:  {loss.item():.4f}")

print("\n── Loss summary ──")
print("Perfect overlap  → Dice Loss close to 0")
print("No overlap       → Dice Loss close to 1")
print("Partial overlap  → Dice Loss between 0 and 1")

── Dice Loss scenarios ──

Scenario 1 — Perfect overlap:
Prediction: [1.0, 1.0, 0.0, 0.0]
Target:     [1.0, 1.0, 0.0, 0.0]
Dice Loss:  0.0000

Scenario 2 — No overlap:
Prediction: [0.0, 0.0, 1.0, 1.0]
Target:     [1.0, 1.0, 0.0, 0.0]
Dice Loss:  0.8000

Scenario 3 — Partial overlap:
Prediction: [1.0, 0.0, 0.0, 0.0]
Target:     [1.0, 1.0, 0.0, 0.0]
Dice Loss:  0.2500

── Loss summary ──
Perfect overlap  → Dice Loss close to 0
No overlap       → Dice Loss close to 1
Partial overlap  → Dice Loss between 0 and 1


In [5]:
# Cell 4 — Model summary and parameter count

import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = UNet(in_channels=1, out_channels=1).to(device)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters()
                       if p.requires_grad)

print(f"── U-Net Parameter Count ──")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print(f"\n── Layer breakdown ──")
for name, param in model.named_parameters():
    print(f"{name:40s} → {str(param.shape):30s} → {param.numel():,} params")

print(f"\n── Memory estimate ──")
param_memory = total_params * 4 / 1024 / 1024
print(f"Parameter memory: {param_memory:.2f} MB")
print(f"(float32 = 4 bytes per parameter)")

print(f"\n── U-Net ready for training ──")
print(f"Input:  [batch, 1, 256, 256]")
print(f"Output: [batch, 1, 256, 256]")
print(f"Loss:   Dice Loss")
print(f"✓ Ready for Week 4 training")

── U-Net Parameter Count ──
Total parameters:     1,962,337
Trainable parameters: 1,962,337

── Layer breakdown ──
enc1.conv1.weight                        → torch.Size([16, 1, 3, 3])      → 144 params
enc1.conv1.bias                          → torch.Size([16])               → 16 params
enc1.conv2.weight                        → torch.Size([16, 16, 3, 3])     → 2,304 params
enc1.conv2.bias                          → torch.Size([16])               → 16 params
enc2.conv1.weight                        → torch.Size([32, 16, 3, 3])     → 4,608 params
enc2.conv1.bias                          → torch.Size([32])               → 32 params
enc2.conv2.weight                        → torch.Size([32, 32, 3, 3])     → 9,216 params
enc2.conv2.bias                          → torch.Size([32])               → 32 params
enc3.conv1.weight                        → torch.Size([64, 32, 3, 3])     → 18,432 params
enc3.conv1.bias                          → torch.Size([64])               → 64 params
enc3.conv2.